In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('../data/processed/insurance_claims_engineered.csv')

X = df.drop(columns=['annual_premium', 'annual_premium_log'])
y = df['annual_premium_log']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline_pred_log = np.full_like(y_test, y_train.mean())

y_test_dollars = np.expm1(y_test)
baseline_pred_dollars = np.expm1(baseline_pred_log)

baseline_rmse = root_mean_squared_error(y_test_dollars, baseline_pred_dollars)
baseline_mae = mean_absolute_error(y_test_dollars, baseline_pred_dollars)

print("Baseline Model Performance:")
print(f"Mean Target Value (Log): {y_train.mean():.4f}")
print(f"RMSE: ${baseline_rmse:,.2f}")
print(f"MAE:  ${baseline_mae:,.2f}")

Baseline Model Performance:
Mean Target Value (Log): 9.0943
RMSE: $3,276.70
MAE:  $2,552.50


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score


numeric_cols = ['age', 'bmi', 'alcohol_units_per_week', 'num_chronic_conditions', 
                'deductible_amount', 'num_dependents', 'policy_tenure_years', 
                'region_cost_index', 'annual_income_log', 'prior_claims_amount_log',
                'claims_per_year', 'avg_claim_amount', 'bmi_smoker_interaction',
                'age_chronic_interaction']


categorical_cols = ['gender', 'region', 'smoker', 'coverage_type', 'employment_status']


ordinal_cols = ['plan_tier', 'age_group', 'bmi_category', 'income_bracket']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
        ('ord', ordinal_transformer, ordinal_cols)
    ],
    remainder='passthrough'
)

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)

lr_preds_log = lr_pipeline.predict(X_test)

lr_preds_dollars = np.expm1(lr_preds_log)

lr_rmse = root_mean_squared_error(y_test_dollars, lr_preds_dollars)
lr_mae = mean_absolute_error(y_test_dollars, lr_preds_dollars)

print("LINEAR REGRESSION PERFORMANCE:")
print(f"RMSE: ${lr_rmse:,.2f}")
print(f"MAE:  ${lr_mae:,.2f}")
print(f"Improvement over baseline: ${(baseline_rmse - lr_rmse):,.2f}")

LINEAR REGRESSION PERFORMANCE:
RMSE: $1,071.01
MAE:  $823.51
Improvement over baseline: $2,205.69


In [9]:
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

lr_r2 = r2_score(y_test_dollars, lr_preds_dollars)
print(f"Linear Regression R²: {lr_r2:.4f}")

ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RidgeCV(alphas=np.logspace(-3, 3, 10)))
])

ridge_pipeline.fit(X_train, y_train)
ridge_preds_log = ridge_pipeline.predict(X_test)
ridge_preds_dollars = np.expm1(ridge_preds_log)

ridge_rmse = root_mean_squared_error(y_test_dollars, ridge_preds_dollars)
ridge_mae = mean_absolute_error(y_test_dollars, ridge_preds_dollars)
ridge_r2 = r2_score(y_test_dollars, ridge_preds_dollars)

print("\nRIDGE REGRESSION PERFORMANCE:")
print(f"RMSE: ${ridge_rmse:,.2f}")
print(f"MAE:  ${ridge_mae:,.2f}")
print(f"R²:   {ridge_r2:.4f}")

Linear Regression R²: 0.8912

RIDGE REGRESSION PERFORMANCE:
RMSE: $1,069.81
MAE:  $822.99
R²:   0.8914


In [12]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LassoCV


lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LassoCV(cv=5, random_state=42))
])

lasso_pipeline.fit(X_train, y_train)
lasso_preds_dollars = np.expm1(lasso_pipeline.predict(X_test))

print("\nLASSO REGRESSION PERFORMANCE:")
print(f"RMSE: ${root_mean_squared_error(y_test_dollars, lasso_preds_dollars):,.2f}")
print(f"R²:   {r2_score(y_test_dollars, lasso_preds_dollars):.4f}\n")


feature_names = lasso_pipeline.named_steps['preprocessor'].get_feature_names_out()


ridge_coefs = ridge_pipeline.named_steps['model'].coef_
lasso_coefs = lasso_pipeline.named_steps['model'].coef_


coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Ridge_Coef': ridge_coefs,
    'Lasso_Coef': lasso_coefs
}).sort_values(key=lambda x: abs(x), by='Ridge_Coef', ascending=False)





LASSO REGRESSION PERFORMANCE:
RMSE: $1,060.10
R²:   0.8934



## RANDOM FOREST


In [11]:
from sklearn.ensemble import RandomForestRegressor


rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
rf_preds_dollars = np.expm1(rf_pipeline.predict(X_test))

rf_rmse = root_mean_squared_error(y_test_dollars, rf_preds_dollars)
rf_mae = mean_absolute_error(y_test_dollars, rf_preds_dollars)
rf_r2 = r2_score(y_test_dollars, rf_preds_dollars)

print(f"--- RANDOM FOREST REGRESSOR ---")
print(f"RMSE: ${rf_rmse:,.2f}")
print(f"MAE:  ${rf_mae:,.2f}")
print(f"R²:   {rf_r2:.4f}\n")

rf_importances = rf_pipeline.named_steps['model'].feature_importances_

rf_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_importances
}).sort_values(by='Importance', ascending=False)


--- RANDOM FOREST REGRESSOR ---
RMSE: $1,143.80
MAE:  $851.75
R²:   0.8759

